# Notebook 02 — DAG-Informed Causal Analysis

**Companion to:** *Reducing Scrap in Injection Molding: A DAG-Informed Causal Decision Analysis*

This notebook implements the causal identification and estimation strategy from Sections 2–4
of the paper. It produces the forest plot (Figure 1), the cooling-time sign-reversal visualization
(Figure 2), and the warpage interaction heatmap (Figure 3).

**Reproduction notes are inline wherever this repo differs from the paper.**


## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

from src.utils import (load_data, PROCESS_LEVERS, PLANNING_LEVER,
                        ADJUSTMENT_SETS, FIXED_EFFECTS, CONFOUNDERS,
                        MEDIATORS, PLANNING_LEVER, BATCH_QUALITY)
from src.causal_helpers import estimate_adjusted_effect, estimate_all_effects
from src.plotting import plot_cooling_sign_reversal, plot_forest, plot_warpage_interaction

pd.set_option("display.float_format", "{:.3f}".format)
df = load_data()
print(f"Loaded: {df.shape}")


## 2. Variable Classification (Paper §2.1)

The ontology partitions columns into five causal roles. This determines adjustment sets and
prevents conditioning on mediators (which would block causal pathways and yield direct-effect
estimates instead of total-effect estimates).


In [ ]:
roles = pd.DataFrame([
    ("Controllable process levers",  str(PROCESS_LEVERS)),
    ("Planning-level lever",         str(PLANNING_LEVER)),
    ("Confounders",                  str(CONFOUNDERS)),
    ("Mediators (NEVER in adj. set)", str(MEDIATORS)),
    ("Fixed effects (absorbed)",     str(FIXED_EFFECTS)),
    ("Batch quality covariate",      str(BATCH_QUALITY)),
], columns=["Role", "Variables"])
roles.style.set_properties(**{"text-align": "left"})


**Mediator exclusion rule:** When estimating a lever's *total* effect, mediators
(`resin_moisture_pct`, `calibration_drift_index`, `tool_wear_index`) must be excluded from
the adjustment set Z. Including them would block the causal pathway from lever to scrap,
giving a controlled *direct* effect estimate — not the total effect we want.

Note: `clamp_force_kn` is not listed as a controllable lever in the paper (§2.1); it appears
as a covariate in the adjustment set for `injection_pressure_bar` (Table 2).


## 3. Identification Strategy

In [ ]:
# Adjustment sets per lever — exactly as specified in paper Table 2
adj_df = pd.DataFrame([
    {"Lever": k, "Adjustment set (beyond fixed effects)": ", ".join(v)}
    for k, v in ADJUSTMENT_SETS.items()
])
adj_df


The target estimand is the Average Causal Effect (ACE):

```
ACE(T, t1, t0) = E[Y | do(T=t1)] - E[Y | do(T=t0)]
```

Under the **backdoor criterion**, this is identified by conditioning on a set Z that blocks
all non-causal paths from T to Y and contains no descendants of T (paper §2.3, eqn 2).

Estimation method: **adjusted linear regression** (paper §3.2, eqn 4), fit separately for
each lever with its own DAG-prescribed adjustment set plus machine/mold/variant/shift FEs.
All continuous features are z-scored so β_std is comparable across levers.


## 4. Adjusted Effect Estimates

In [ ]:
print("Estimating adjusted effects (n_bootstrap=300)…")
effects_df = estimate_all_effects(
    df, levers=list(ADJUSTMENT_SETS.keys()), n_bootstrap=300
)


In [ ]:
# Display table
display_df = effects_df[["lever","beta_std","ci_lo","ci_hi","beta_unstd"]].copy()
display_df.columns = ["Lever", "β (std)", "CI lo", "CI hi", "β̃ (p.p./unit)"]
display_df = display_df.sort_values("β (std)")

print("NOTE ON COEFFICIENTS:")
print("  β (std):       Frisch-Waugh coeff from z-scored lever / unscaled outcome.")
print("                 Comparable across levers; matches paper Table 2 values.")
print("  β̃ (p.p./unit): Direct OLS in original units. For cooling_time_s: p.p./s.")
print("                 Matches paper's β̃ = -0.41 p.p./s for cooling.\n")
display_df.round(3)


In [ ]:
# Cooling worked example
r_c = effects_df[effects_df["lever"]=="cooling_time_s"].iloc[0]
print(f"Cooling time — paper vs repo:")
print(f"  β (std):         paper=-1.75   repo={r_c['beta_std']:+.3f}")
print(f"  β̃ (p.p./s):      paper=-0.41   repo={r_c['beta_unstd']:+.4f}")
print()
print(f"  Linear estimate of +1.5s intervention:")
print(f"  ΔŶ = β̃ × Δt = {r_c['beta_unstd']:.4f} × 1.5 = {r_c['beta_unstd']*1.5:.4f} p.p.")
print(f"  Paper (eqn 7):  -0.41 × 1.5 = -0.615 p.p.")
print()
print("NOTE: β(std) and β̃ are numerically inconsistent with paper eqn 5")
print("  (β_std = β̃ × σT/σY) because the outcome is not z-scored in the")
print("  Frisch-Waugh step. The same inconsistency is present in the paper.")
print("  Both quantities individually match their respective paper values.")


## 5. Forest Plot (Figure 1)

In [ ]:
fig = plot_forest(effects_df[["lever","beta_std","ci_lo","ci_hi"]].rename(
    columns={"lever":"variable"}))
plt.show()
print("Saved to figures/forest_plot.png")


**Key result:** `cooling_time_s` is the only lever where the sign reverses between
the raw correlation and the adjusted estimate. Every other lever's DAG-adjusted direction
aligns with its raw correlation.

**Note on CI width:** CIs use row-wise (i.i.d.) bootstrap. A machine-clustered bootstrap
would produce wider intervals. Signs and rankings are robust; treat magnitudes as approximate.


## 6. The Cooling-Time Sign Change (Paper §4.3)

In [ ]:
from scipy.stats import pearsonr
rho_raw, _ = pearsonr(df["cooling_time_s"], df["scrap_rate_pct"])
from sklearn.linear_model import LinearRegression

X = df[["mold_temperature_c"]].values
res_y = df["scrap_rate_pct"].values - LinearRegression().fit(X, df["scrap_rate_pct"].values).predict(X)
res_c = df["cooling_time_s"].values  - LinearRegression().fit(X, df["cooling_time_s"].values).predict(X)
rho_partial, _ = pearsonr(res_c, res_y)

print(f"Raw correlation (cooling vs scrap):        ρ = {rho_raw:+.3f}  (paper: +0.28)")
print(f"Partial ρ (after adjusting mold temp):     ρ = {rho_partial:+.3f}  (paper: -0.37)")
print(f"Full adjusted β (std):                     β = {r_c['beta_std']:+.3f}  (paper: -1.75)")


In [ ]:
fig = plot_cooling_sign_reversal(df)
plt.show()


**Mechanism — reverse causation:**  Operators already extend cooling reactively when
they observe high mold temperatures (elevated thermal stress). The raw positive correlation
captures this compensation, not a direct effect of cooling on scrap. Adjusting for
`mold_temperature_c` removes this confounding and reveals the protective direction.

This is a DAG-informed estimate under the stated identification assumptions — not an
experimental coefficient. The sign is robust across all four estimation methods; the
magnitude depends on the model.


## 7. Warpage Interaction (Paper §4.4, Table 3 / Figure 3)

In [ ]:
fig = plot_warpage_interaction(df)
plt.show()


In [ ]:
# Verify cross-tab matches paper Table 3
warp = df[df["defect_type"]=="warpage"].copy()
warp["mold_bin"] = pd.cut(warp["mold_temperature_c"],
    bins=[-np.inf,65,75,np.inf], labels=["≤65°C","65–75°C",">75°C"])
warp["cool_bin"] = pd.cut(warp["cooling_time_s"],
    bins=[-np.inf,12,18,np.inf], labels=["≤12s","12–18s",">18s"])

pivot_mean = warp.pivot_table("scrap_rate_pct","mold_bin","cool_bin","mean").round(2)
pivot_n    = warp.pivot_table("scrap_rate_pct","mold_bin","cool_bin","count").astype(int)

print("Mean scrap % (warpage subset) — should match paper Table 3 exactly:")
print(pivot_mean.to_string())
print()
print("Counts:")
print(pivot_n.to_string())


**Key number:** At mold > 75 °C, moving from cooling ≤12 s to cooling > 18 s reduces
mean scrap from **8.2% to 5.1% (−3.1 p.p.)** in the warpage subset.

The interaction confirms: the protective effect of cooling is amplified at high mold
temperatures — precisely where the thermal problem is worst.


## 8. Effects Disclosed but Not Actioned (Paper §4.5)

In [ ]:
not_actioned = pd.DataFrame([
    ("operator_experience_level", "Residual assignment bias — senior operators on harder shifts/variants. "
     "Not a direct causal effect on scrap."),
    ("hold_pressure_bar",         "β ≈ −0.09 (mild protection) but 95% CI narrowly crosses zero. "
     "Tracked in Table 2 but excluded from action package on significance grounds."),
], columns=["Variable", "Reason not actioned"])
not_actioned


## Summary

| Lever | β (std) | Direction | Action |
|---|---|---|---|
| `cooling_time_s` | −1.74 | Extend | ✅ High confidence |
| `mold_temperature_c` | +0.88 | Cap at 78 °C | ✅ Med–High |
| `injection_pressure_bar` | +0.32 | Reduce (conditional) | ✅ Medium |
| `dryer_dewpoint_c` | +0.10 | Lower (conditional) | ✅ Medium |
| `maintenance_days_since_last` | +0.10 | Cap at 14 days | ✅ Medium |
| `hold_pressure_bar` | −0.09 | — | ⚠️ CI crosses zero |
| `barrel_temperature_c` | −0.14 | — | ⚠️ Small effect |
| `screw_speed_rpm` | +0.00 | — | ❌ No effect |
| `operator_experience_level` | +0.12 | — | ❌ Assignment bias |

Next: Notebook 03 sizes the interventions via GBR counterfactual simulation.
